In [ ]:
!git clone https://github.com/t-ravnushkin/generalized-toric-gpu-search.git
import sys, os
sys.path.insert(0, "/kaggle/working/generalized-toric-gpu-search")
os.chdir("/kaggle/working")

# GF(8) Toric Code Champion Search — Kaggle (NVIDIA GPU)

CUDA-native implementation: one block per set, threads split polynomial space.
Falls back to OpenCL if CuPy is unavailable.

In [ ]:
# CuPy is pre-installed on Kaggle T4/P100/A100 — no pip needed.
# PyOpenCL is kept as fallback only.
try:
    import cupy as cp
    _USE_CUDA = True
    print(f"CuPy {cp.__version__} available — using CUDA kernels")
except ImportError:
    _USE_CUDA = False
    !pip install pyopencl -q
    print("CuPy not found — using OpenCL kernels")

import sys, os
sys.path.insert(0, "/kaggle/working/generalized-toric-gpu-search")
os.chdir("/kaggle/working")

In [ ]:
if _USE_CUDA:
    import cupy as cp
    n = cp.cuda.runtime.getDeviceCount()
    for i in range(n):
        p = cp.cuda.runtime.getDeviceProperties(i)
        print(f"Device {i}: {p['name'].decode()}  "
              f"({p.get('multiProcessorCount',0)} SMs, "
              f"{p['totalGlobalMem']//1024**3} GB VRAM)")
else:
    import pyopencl as cl
    for p in cl.get_platforms():
        for d in p.get_devices():
            kind = 'GPU' if d.type == cl.device_type.GPU else 'CPU'
            print(f"{p.name}  |  {d.name}  |  {kind}")

In [ ]:
from precompute import build_eval_matrix, bounding_cube_points
from champion_search import find_latest_results, load_results, check_set, global_batched_bfs, idx
from pathlib import Path
from datetime import datetime

lattice = bounding_cube_points()

# ── kernel selection ──────────────────────────────────────────────────────
# Options:
#   "cuda_bp"  — bit-parallel CUDA (recommended; ~14× fewer ops/poly vs v1)
#   "cuda"     — M-cache CUDA (v1, fewer changes from OpenCL)
#   "opencl"   — original OpenCL fallback
KERNEL = "cuda_bp"

if KERNEL in ("cuda_bp", "cuda"):
    M = build_eval_matrix()
    if KERNEL == "cuda_bp":
        from kernel_cuda_bp import init_cuda_oracles_bp as _init
        oracles, sm_count = _init(M)
    else:
        from kernel_cuda import init_cuda_oracles as _init
        oracles, sm_count = _init(M)
    n_gpus = len(oracles)
else:
    from precompute import init_opencl_all
    from kernel import DistanceOracle
    import pyopencl as cl, os
    os.environ.setdefault("PYOPENCL_CTX", "0")
    gpu_contexts, _ = init_opencl_all()
    oracles = [DistanceOracle(ctx, queue, M_buf) for ctx, queue, M_buf in gpu_contexts]
    sm_count = oracles[0].ctx.devices[0].max_compute_units
    n_gpus = len(oracles)

ADAPTIVE_BATCH = sm_count * 1024 * 8 * n_gpus
print(f"\n{n_gpus} GPU(s) ready  [{KERNEL}]")
print(f"Batch size: {ADAPTIVE_BATCH:,}  ({sm_count} SMs × {n_gpus} GPU(s))")

In [ ]:
# ── Quick benchmark: compare kernel variants on one BFS level ─────────────
# Set k_bench to the level you want to measure (5–8 is a good range).
# Runs each kernel on the same random batch and prints wall-clock times.

import time, random

def _bench(oracles_list, batch, td, label):
    t0 = time.perf_counter()
    for oracle in oracles_list:
        oracle.max_zeros_batch(batch, td)
    dt = time.perf_counter() - t0
    print(f"  {label:20s}: {dt*1000:.1f} ms  ({len(batch)} sets, k={len(batch[0])})")

k_bench = 6
td_bench = 10
n_bench  = 4096   # sets to benchmark

random.seed(42)
bench_batch = [sorted(random.sample(range(49), k_bench)) for _ in range(n_bench)]

from precompute import build_eval_matrix
M_ref = build_eval_matrix()

from kernel_cuda    import DistanceOracleCUDA
from kernel_cuda_bp import DistanceOracleCUDABP

# compile on device 0
oracle_v1 = DistanceOracleCUDA(0, M_ref)
oracle_bp = DistanceOracleCUDABP(0, M_ref)

# warm-up
oracle_v1.max_zeros_batch(bench_batch[:64], td_bench)
oracle_bp.max_zeros_batch(bench_batch[:64], td_bench)

print(f"Benchmark: k={k_bench}, {n_bench} random sets, target_d={td_bench}")
_bench([oracle_v1], bench_batch, td_bench, "cuda v1 (M-cache)")
_bench([oracle_bp], bench_batch, td_bench, "cuda bp (bit-parallel)")

## Codetables.de — best known bounds for [49, k] codes over GF(8)

These are the targets our toric codes are racing against.
A result matching or beating the table entry would be a new record.

In [ ]:
from codetables import bounds_for_n
import pandas as pd

targets = bounds_for_n(49)   # fetched once, cached in .gf8_bounds_cache.txt

df_bounds = pd.DataFrame(
    [(k, d) for k, d in sorted(targets.items()) if k <= 15],
    columns=["k", "best_known_d"]
)
display(df_bounds)

In [ ]:
results_file = find_latest_results()
if results_file is None:
    results_file = Path(f"champions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    print(f"Starting fresh: {results_file}")
else:
    print(f"Resuming from: {results_file}  ({len(load_results(results_file))} records)")

## Part A — Structured geometric candidates

In [ ]:
already_done = (
    {rec["name"] for rec in load_results(results_file) if "name" in rec}
    if results_file.exists() else set()
)

circle_8 = sorted([idx(0,1), idx(0,6), idx(1,0), idx(2,2), idx(2,5), idx(5,2), idx(5,5), idx(6,0)])
check_set("circle_conic_k8", circle_8, oracles[0], lattice, results_file, already_done=already_done)

parabola_7 = sorted([idx(t, (t*t) % 7) for t in range(7)])
check_set("parabola_conic_k7", parabola_7, oracles[0], lattice, results_file, already_done=already_done)

## Part B — Global BFS guided by codetables.de bounds

The table gives the best known d for *any* linear [49, k] code over GF(8).
Toric codes are a restricted subclass and typically fall 0–3 points short of those bounds.
`margin` subtracts that gap so the BFS doesn't prune everything on the first level.
Increase it if level k=2 still shows 0 survivors; decrease it to search more aggressively.

In [ ]:
import time

margin = 3  # raise if BFS dies at k=2 (target too high); lower to search harder
adjusted = {k: max(d - margin, 1) for k, d in targets.items()}
print("Adjusted per-k targets:", {k: adjusted[k] for k in sorted(adjusted) if k <= 10})

t0 = time.perf_counter()
global_batched_bfs(
    target_distance=adjusted,
    oracles=oracles,
    lattice=lattice,
    results_file=results_file,
    max_k=10,
    resume=True,
    batch_size=ADAPTIVE_BATCH,
)
print(f"\nTotal BFS time: {time.perf_counter() - t0:.1f}s")

## Results vs. best known bounds

In [ ]:
records = [r for r in load_results(results_file) if r.get("type") != "level_complete"]
df = pd.DataFrame(records)[["name", "k", "min_distance", "indices"]]
df["best_known_d"] = df["k"].map(targets)
df["gap"] = df["best_known_d"] - df["min_distance"]
df["new_record"] = df["gap"] < 0
df.sort_values(["k", "min_distance"], ascending=[True, False], inplace=True)
df.reset_index(drop=True, inplace=True)
display(df)